# Knee Exo — batch GT vs logged model

Batch comparison for all `*_knee_*_exo_on` telemetry trials.

- **Sync**: GPIO falling-edge alignment (telemetry vs mocap jet)
- **GT**: Vicon ID/mass − `cmd_R`/mass, zero-phase 6 Hz LPF
- **Model**: logged `model_out_nmpkg_raw` from telemetry NPZ (no re-inference), zero-phase 6 Hz LPF before metrics/plots
- **Oracle**: offline TCN replay with Vicon IK or encoder + causal dθ/dt vs same GT
- **Metrics / plots**: RMSE and R² after removing first/last 10 s post-sync (`TRIM_START_SEC` / `TRIM_END_SEC`)


In [ ]:
import io
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
TELEMETRY_ROOT = PROJECT_ROOT

EXO_KIND = 'knee-exo'
JOINT = 'knee_angle_r'
JOINT_LABEL = 'Knee R'
MOMENT_COL = 'knee_angle_r_moment'
MOCAP_FS_HZ = 1000.0
LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
GT_LPF_MODE = 'zero_phase'
MODEL_LPF_MODE = 'zero_phase'
TELEMETRY_PATTERN = '*_knee_*_exo_on.npz'
GT_METRIC_LABEL = 'GT (ID/mass − cmd_R/mass) vs logged model'
PAPER_SUBDIR = 'knee_exo'

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}
SUBJECT_MASS_KG = {
    'ab01_jinwoo': 88.0, 'ab02_oscar': 71.1, 'ab03_ilseung': 84.4, 'ab04_changseob': 74.0,
    'ab05_maria': 55.0, 'ab06_jimin': 82.6, 'ab07_amy': 51.3, 'ab08_seokhyun': 71.9,
}

CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / 'compare_processed_knee_exo_id.npz'
METRICS_CSV_PATH = CACHE_DIR / 'compare_processed_knee_exo_id_metrics.csv'

LOAD_FROM_CACHE = False
SAVE_TO_CACHE = True

# Post-sync trim applied to metrics and plots (set ends to 0.0 to disable).
TRIM_START_SEC = 10.0
TRIM_END_SEC = 10.0

# Set to None to include all subjects.
EXCLUDE_SUBJECTS = None

print(f'Exo: {EXO_KIND} | joint: {JOINT_LABEL}')
print(f'Processed root: {PROCESSED_ROOT}')
print(f'Cache: {CACHE_PATH}')
print(f'LOAD_FROM_CACHE={LOAD_FROM_CACHE} | SAVE_TO_CACHE={SAVE_TO_CACHE}')
print(f'EXCLUDE_SUBJECTS={EXCLUDE_SUBJECTS}')


In [ ]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    return rmse, float(1.0 - ss_res / (ss_tot + 1e-12))


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def _subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def subject_dir_from_stem(stem: str) -> Path:
    token = _subject_token(stem)
    name = SUBJECT_TOKEN_TO_DIR[token]
    p = PROCESSED_ROOT / name
    if not p.is_dir():
        raise FileNotFoundError(p)
    return p


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def trial_meta_from_stem(stem: str) -> Dict[str, object]:
    token = _subject_token(stem)
    subject = SUBJECT_TOKEN_TO_DIR[token]
    cond, speed = trial_cond_speed(stem)
    condition = f'{cond}_{speed}'
    speed_mps = float(speed.replace('mps', '').replace('p', '.'))
    return {
        'subject': subject,
        'task': cond,
        'speed': speed,
        'condition': condition,
        'speed_mps': speed_mps,
        'trial_key': f'{subject}::{condition}',
    }


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines='skip')
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    return np.arange(len(df)) / fs, df['jet'].to_numpy(dtype=float)


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    arr = np.asarray(gpio, dtype=np.float64)
    g_range = arr.max() - arr.min()
    return arr if g_range <= 0 else (arr - arr.min()) / g_range


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    above = np.asarray(signal, dtype=np.float64) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError('No GPIO key in npz')


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    subj_dir = subject_dir_from_stem(trial_stem)
    return {
        'npz': TELEMETRY_ROOT / f'{trial_stem}.npz',
        'mocap': subj_dir / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv',
        'id': subj_dir / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto',
        'cond': cond,
        'speed': speed,
        'subject_dir': subj_dir,
    }


def load_gpio_sync_data(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap', 'id'):
        if not paths[key].exists():
            raise FileNotFoundError(f'Missing {key}: {paths[key]}')

    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = np.asarray(d['time'], dtype=np.float64) if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]

    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)

    return {
        'trial': trial_stem,
        'sync_method': 'gpio_falling_edge',
        'paths': paths,
        'npz': d,
        'gpio_key': gpio_key,
        'offset_s': offset_s,
        'idx_exo': idx_exo,
        'idx_mocap': idx_mocap,
        'fs_npz_hz': infer_fs_hz(t_raw),
        'fs_mocap_hz': infer_fs_hz(t_mocap, default_fs=MOCAP_FS_HZ),
        't_npz': t_raw,
        'gpio_npz': gpio,
        't_mocap': t_mocap,
        'gpio_mocap_raw': gpio_mocap,
        'gpio_mocap_norm': normalize_gpio(gpio_mocap),
        't_npz_aligned': t_raw + float(offset_s) if offset_s is not None else t_raw.copy(),
    }


def is_excluded_subject(subject_name: str) -> bool:
    return EXCLUDE_SUBJECTS is not None and subject_name in EXCLUDE_SUBJECTS


def valid_telemetry_stem(stem: str) -> bool:
    token = _subject_token(stem)
    if token not in SUBJECT_TOKEN_TO_DIR:
        return False
    return not is_excluded_subject(SUBJECT_TOKEN_TO_DIR[token])


def analysis_trim_mask(
    t: np.ndarray,
    trim_start_s: float = TRIM_START_SEC,
    trim_end_s: float = TRIM_END_SEC,
) -> np.ndarray:
    """Post-sync mask: drop first/last N seconds of each trial."""
    t_rel = np.asarray(t, dtype=np.float64) - np.nanmin(t)
    t_end = float(np.nanmax(t_rel))
    return (t_rel >= float(trim_start_s)) & (t_rel <= t_end - float(trim_end_s))


def model_out_nmpkg_lpf(raw: np.ndarray, fs_hz: float) -> np.ndarray:
    return lpf_nan(raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, MODEL_LPF_MODE)


def enrich_model_out_nmpkg(wave: Dict) -> Dict:
    if 'model_out_nmpkg' in wave:
        return wave
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(wave['t'])))
    out = dict(wave)
    out['fs_hz'] = fs_hz
    out['model_out_nmpkg'] = model_out_nmpkg_lpf(out['model_out_nmpkg_raw'], fs_hz)
    return out


def save_processed_exo_cache(trial_data: Dict[str, Dict], path: Path = CACHE_PATH) -> None:
    if not trial_data:
        raise RuntimeError('TRIAL_DATA is empty')
    payload = {'trial_keys': np.array(sorted(trial_data.keys()), dtype=object)}
    for trial_key, d in trial_data.items():
        p = str(trial_key)
        payload[f'{p}__t'] = np.asarray(d['t'], dtype=np.float64)
        payload[f'{p}__gt_nmpkg'] = np.asarray(d['gt_nmpkg'], dtype=np.float64)
        payload[f'{p}__model_out_nmpkg_raw'] = np.asarray(d['model_out_nmpkg_raw'], dtype=np.float64)
        payload[f'{p}__model_out_nmpkg'] = np.asarray(d['model_out_nmpkg'], dtype=np.float64)
        payload[f'{p}__meta'] = np.array([
            d['joint'], d['moment_col'], d['mass_kg'], d['offset_s'],
            d['applied_key'], d['gpio_key'], d['model_out_key'], d['trial_key'],
        ], dtype=object)
    np.savez_compressed(str(path), **payload)
    print(f'Saved {len(trial_data)} trials → {path}')


def load_processed_exo_cache(path: Path = CACHE_PATH) -> Dict[str, Dict]:
    if not path.exists():
        raise FileNotFoundError(f'Cache not found: {path}')
    data = np.load(str(path), allow_pickle=True)
    trial_data: Dict[str, Dict] = {}
    for trial_key in data['trial_keys']:
        p = str(trial_key)
        meta = data[f'{p}__meta']
        wave = {
            'trial': p,
            'joint': str(meta[0]),
            'moment_col': str(meta[1]),
            'mass_kg': float(meta[2]),
            'offset_s': float(meta[3]),
            'applied_key': str(meta[4]),
            'gpio_key': str(meta[5]),
            'model_out_key': str(meta[6]),
            'trial_key': str(meta[7]),
            't': np.asarray(data[f'{p}__t'], dtype=np.float64),
            'gt_nmpkg': np.asarray(data[f'{p}__gt_nmpkg'], dtype=np.float64),
            'model_out_nmpkg_raw': np.asarray(data[f'{p}__model_out_nmpkg_raw'], dtype=np.float64),
        }
        if f'{p}__model_out_nmpkg' in data.files:
            wave['model_out_nmpkg'] = np.asarray(data[f'{p}__model_out_nmpkg'], dtype=np.float64)
        trial_data[p] = enrich_model_out_nmpkg(wave)
    print(f'Loaded {len(trial_data)} trials from {path}')
    return trial_data

def extract_model_out_nmpkg_raw(npz) -> Tuple[np.ndarray, str]:
    for k in ('model_out_nmpkg_raw', 'model_out_nmpkg'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No model output key; available: {sorted(npz.files)}')


def extract_applied_cmd_nm(npz) -> Tuple[np.ndarray, str]:
    for k in ('cmd_R', 'cmd_L'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No cmd_R/cmd_L; available: {sorted(npz.files)}')


def load_moment_waveforms(trial_stem: str, sync: Dict) -> Dict:
    """GPIO-synced GT vs logged model_out_nmpkg_raw."""
    paths = sync['paths']
    mass = SUBJECT_MASS_KG[_subject_token(trial_stem)]
    d = sync['npz']

    applied_nm, applied_key = extract_applied_cmd_nm(d)
    model_out_raw, model_out_key = extract_model_out_nmpkg_raw(d)

    n = min(len(sync['t_npz']), len(applied_nm), len(model_out_raw))
    t_aligned = sync['t_npz_aligned'][:n].astype(np.float64)
    applied_nm = applied_nm[:n]
    model_out_nmpkg_raw = np.asarray(model_out_raw[:n], dtype=np.float64)
    fs_hz = infer_fs_hz(sync['t_npz'][:n])

    cols, id_data = read_sto(paths['id'])
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(MOMENT_COL)]

    id_nm_raw = np.interp(t_aligned, t_id, id_moment_nm, left=np.nan, right=np.nan)
    id_nmpkg_raw = id_nm_raw / mass
    applied_nmpkg_raw = applied_nm / mass

    net_raw_nmpkg = id_nmpkg_raw - applied_nmpkg_raw
    gt_nmpkg = lpf_nan(net_raw_nmpkg, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, GT_LPF_MODE)
    model_out_nmpkg = model_out_nmpkg_lpf(model_out_nmpkg_raw, fs_hz)

    meta = trial_meta_from_stem(trial_stem)
    return {
        'trial': trial_stem,
        'trial_key': meta['trial_key'],
        'joint': JOINT,
        't': t_aligned,
        'gt_nmpkg': gt_nmpkg,
        'model_out_nmpkg_raw': model_out_nmpkg_raw,
        'model_out_nmpkg': model_out_nmpkg,
        'applied_key': applied_key,
        'model_out_key': model_out_key,
        'mass_kg': mass,
        'moment_col': MOMENT_COL,
        'fs_hz': fs_hz,
    }




def _fill_nan_1d(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def sync_to_wave_t(
    t_src: np.ndarray,
    y_src: np.ndarray,
    wave: Dict,
    *,
    t_src_on_npz_clock: bool = False,
) -> np.ndarray:
    """Resample onto GPIO-aligned `wave['t']` (same clock as ID GT / logged model)."""
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    t_src = np.asarray(t_src, dtype=np.float64)
    if t_src_on_npz_clock:
        n = int(min(len(t_aligned), len(t_src), len(y_src)))
        t_aligned = t_aligned[:n]
        t_src = t_src[:n]
        y_src = y_src[:n]
        t_ref = t_src + float(wave['offset_s'])
    else:
        t_ref = t_src
    y_sync = np.interp(t_aligned, t_ref, y_src, left=np.nan, right=np.nan)
    return _fill_nan_1d(y_sync)


def process_trial_waveforms(trial_stem: str):
    try:
        sync = load_gpio_sync_data(trial_stem)
    except FileNotFoundError as exc:
        return None, str(exc)
    if sync['offset_s'] is None:
        return None, 'no GPIO falling edge'
    wave = load_moment_waveforms(trial_stem, sync)
    wave['offset_s'] = float(sync['offset_s'])
    wave['gpio_key'] = sync['gpio_key']
    return wave, None

print('Helpers ready.')


## 1. Batch processing (GPIO sync)

In [ ]:
CANDIDATES = sorted(
    p for p in TELEMETRY_ROOT.glob(TELEMETRY_PATTERN)
    if valid_telemetry_stem(p.stem)
)
print(f'Found {len(CANDIDATES)} telemetry files')

WARNINGS: List[str] = []
TRIAL_DATA: Dict[str, Dict] = {}

if LOAD_FROM_CACHE:
    TRIAL_DATA = load_processed_exo_cache(CACHE_PATH)
else:
    for npz_path in CANDIDATES:
        stem = npz_path.stem
        try:
            wave, err = process_trial_waveforms(stem)
            if wave is None:
                WARNINGS.append(f'[WARN] {stem}: {err}')
                print(f'SKIP {stem}: {err}')
                continue
            TRIAL_DATA[stem] = wave
            print(f"OK  {stem} | offset={wave['offset_s']:+.3f}s | n={len(wave['t'])}")
        except Exception as exc:
            WARNINGS.append(f'[WARN] {stem}: {exc}')
            print(f'FAIL {stem}: {exc}')

    if SAVE_TO_CACHE and TRIAL_DATA:
        save_processed_exo_cache(TRIAL_DATA)

def filter_excluded_trials(trial_data: Dict[str, Dict]) -> Dict[str, Dict]:
    if EXCLUDE_SUBJECTS is None:
        return trial_data
    kept = {
        stem: wave for stem, wave in trial_data.items()
        if not is_excluded_subject(SUBJECT_TOKEN_TO_DIR[_subject_token(stem)])
    }
    dropped = len(trial_data) - len(kept)
    if dropped:
        print(f'Excluded subjects {sorted(EXCLUDE_SUBJECTS)} → dropped {dropped} trial(s)')
    return kept


TRIAL_DATA = filter_excluded_trials(TRIAL_DATA)
TRIAL_DATA = {stem: enrich_model_out_nmpkg(wave) for stem, wave in TRIAL_DATA.items()}

for w in WARNINGS:
    print(w)
print(f'\nLoaded {len(TRIAL_DATA)} trials')


## 2. Metrics summary

In [ ]:
if not TRIAL_DATA:
    raise RuntimeError('No trials loaded.')

metrics_rows = []
for stem, wave in sorted(TRIAL_DATA.items()):
    meta = trial_meta_from_stem(stem)
    m = analysis_trim_mask(wave['t'])
    rmse, r2 = rmse_r2(wave['gt_nmpkg'][m], wave['model_out_nmpkg'][m])
    metrics_rows.append({
        'trial': stem,
        'trial_key': meta['trial_key'],
        'subject': meta['subject'],
        'task': meta['task'],
        'condition': meta['condition'],
        'rmse_nmpkg': rmse,
        'r2_nmpkg': r2,
        'offset_s': wave['offset_s'],
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(METRICS_CSV_PATH, index=False)
print(f'Saved metrics → {METRICS_CSV_PATH}')
display(metrics_df)
if metrics_df['rmse_nmpkg'].notna().any():
    print(
        f"Overall: RMSE={metrics_df['rmse_nmpkg'].mean():.4f} N·m/kg | "
        f"R²={metrics_df['r2_nmpkg'].mean():.4f}"
    )


## 3. Interactive QC

In [ ]:
GPIO_PALETTE = {'mocap': '#90A4AE', 'telemetry': '#FF9800'}

def draw_gpio_sync(sync: Dict, window_s: float = 4.0) -> None:
    t_edge_mocap = float(sync['t_mocap'][sync['idx_mocap']])
    t_edge_npz = float(sync['t_npz'][sync['idx_exo']])
    t0, t1 = t_edge_mocap - 0.5, t_edge_mocap + window_s
    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
    m_npz = (sync['t_npz'] >= t_edge_npz - 0.5) & (sync['t_npz'] <= t_edge_npz + window_s)
    m_mocap = (sync['t_mocap'] >= t0) & (sync['t_mocap'] <= t1)
    axs[0].plot(sync['t_npz'][m_npz], sync['gpio_npz'][m_npz], color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label=f"Telemetry ({sync['gpio_key']})")
    axs[0].plot(sync['t_mocap'][m_mocap], sync['gpio_mocap_norm'][m_mocap], color=GPIO_PALETTE['mocap'], lw=1.2, label='Mocap jet (norm)')
    axs[0].axvline(t_edge_npz, color=GPIO_PALETTE['telemetry'], ls=':')
    axs[0].axvline(t_edge_mocap, color=GPIO_PALETTE['mocap'], ls=':')
    axs[0].set_ylabel('Amplitude (a.u.)'); axs[0].set_title('Before sync'); axs[0].legend(); axs[0].grid(alpha=0.25)
    m_a_npz = (sync['t_npz_aligned'] >= t0) & (sync['t_npz_aligned'] <= t1)
    axs[1].plot(sync['t_mocap'][m_mocap], sync['gpio_mocap_norm'][m_mocap], color=GPIO_PALETTE['mocap'], lw=1.2, label='Mocap jet (norm)')
    axs[1].plot(sync['t_npz_aligned'][m_a_npz], sync['gpio_npz'][m_a_npz], color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label='Telemetry shifted')
    axs[1].axvline(t_edge_mocap, color='black', ls=':')
    axs[1].set_xlabel('Mocap time (s)'); axs[1].set_ylabel('Amplitude (a.u.)')
    axs[1].set_title(f'After sync | offset {sync["offset_s"]:+.4f} s')
    axs[1].legend(); axs[1].grid(alpha=0.25)
    fig.suptitle(f"{sync['trial']} | GPIO sync", y=1.01)
    fig.tight_layout(); plt.show()

gpio_trial_dd = widgets.Dropdown(options=sorted(TRIAL_DATA), description='Trial:')
gpio_window = widgets.FloatSlider(value=4.0, min=1.0, max=15.0, step=0.5, description='Window (s):')
gpio_out = widgets.Output()

def _draw_gpio(*_):
    with gpio_out:
        gpio_out.clear_output(wait=True)
        draw_gpio_sync(load_gpio_sync_data(gpio_trial_dd.value), gpio_window.value)

gpio_trial_dd.observe(_draw_gpio, names='value')
gpio_window.observe(_draw_gpio, names='value')
display(widgets.VBox([widgets.HBox([gpio_trial_dd, gpio_window]), gpio_out]))
_draw_gpio()


In [ ]:
res_out = widgets.Output()
res_trial_dd = widgets.Dropdown(options=sorted(TRIAL_DATA), description='Trial:')
res_slider = widgets.FloatRangeSlider(description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px'))

def draw_gt_residual(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    t0, t1 = t_window
    gt = wave['gt_nmpkg']
    model = wave['model_out_nmpkg']
    m = analysis_trim_mask(wave['t']) & (t_rel >= t0) & (t_rel <= t1) & np.isfinite(gt) & np.isfinite(model)
    rmse, r2 = rmse_r2(gt[m], model[m])
    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axs[0].plot(t_rel[m], gt[m], color='#1e88e5', lw=2.0, label='GT')
    axs[0].plot(t_rel[m], model[m], color='#e53935', lw=1.6, ls='--', label='Logged model (6 Hz LPF)')
    axs[0].set_ylabel('N·m/kg')
    axs[0].set_title(f"RMSE={rmse:.4f} | R²={r2:.4f} | offset={wave['offset_s']:+.3f}s")
    axs[0].legend(); axs[0].grid(alpha=0.25)
    axs[1].plot(t_rel[m], (model - gt)[m], color='#9c27b0', lw=1.4, label='Model − GT')
    axs[1].axhline(0, color='black', ls=':')
    axs[1].set_ylabel('Residual (N·m/kg)'); axs[1].set_xlabel('Time (s)')
    axs[1].legend(); axs[1].grid(alpha=0.25)
    fig.tight_layout()
    with res_out:
        res_out.clear_output(wait=True); plt.show()

def _init_res_slider(trial_key: str) -> None:
    wave = TRIAL_DATA[trial_key]
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t'])
    t_use = t_rel[m] if m.any() else t_rel
    res_slider.min = float(t_use[0]); res_slider.max = float(t_use[-1])
    res_slider.step = max((res_slider.max - res_slider.min) / 500, 1e-3)
    res_slider.value = (res_slider.min, res_slider.max)

def _redraw_res(*_):
    draw_gt_residual(TRIAL_DATA[res_trial_dd.value], res_slider.value)

def _on_res_trial(change):
    _init_res_slider(change['new']); _redraw_res()

res_trial_dd.observe(_on_res_trial, names='value')
res_slider.observe(_redraw_res, names='value')
_init_res_slider(res_trial_dd.value)
display(widgets.VBox([res_trial_dd, res_slider, res_out]))
_redraw_res()


## 4. Offline model replay vs GT

Re-runs the deployed knee TCN (`runs/0512_ik_id_knee_causal_in_zero_out/best_model.pt`) on GPIO-synced trials:

- **Logged model**: on-device output from telemetry NPZ
- **Encoder replay**: GPIO-sync encoder angle onto `wave['t']` → causal input LPF → **causal backward dθ/dt** → TCN → output LPF (same timeline as GT / Vicon oracle)
- **Vicon IK oracle**: GPIO-synced Vicon IK angle + interpolated velocity
- **GT**: Vicon ID/mass − `cmd_R`/mass

All replays use causal input LPF (from `final.yaml`) and 6 Hz zero-phase output LPF.


In [ ]:
import inspect
import sys

import torch
import yaml

if not TRIAL_DATA:
    raise RuntimeError('No trials loaded. Run batch processing first.')

CTRL_CFG_PATH = PROJECT_ROOT / 'knee-exo-ctrl' / 'cfg' / 'final.yaml'
VICON_ORACLE_CKPT = PROJECT_ROOT / 'runs' / '0512_ik_id_knee_causal_in_zero_out' / 'best_model.pt'
MODEL_COMPARE_METRICS_CSV = CACHE_DIR / 'compare_processed_knee_exo_id_model_compare_metrics.csv'

_ctrl_cfg = yaml.safe_load(CTRL_CFG_PATH.read_text()) if CTRL_CFG_PATH.is_file() else {}
VICON_ANGLE_LPF_HZ = float(_ctrl_cfg.get('angle_lpf_hz', 6.0))
VICON_ANGLE_LPF_ORDER = int(_ctrl_cfg.get('angle_lpf_order', 4))
VICON_VEL_LPF_HZ = float(_ctrl_cfg.get('vel_lpf_hz', 10.0))
VICON_VEL_LPF_ORDER = int(_ctrl_cfg.get('vel_lpf_order', 4))

sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN  # noqa: E402


class _CausalLowPass:
    """Streaming causal LPF (same structure as knee `CascadeUni`)."""

    def __init__(self, fs_hz: float, cutoff_hz: float, order: int = 4):
        self.order = max(1, int(order))
        if cutoff_hz <= 0.0:
            self.alpha = 1.0
        else:
            dt = 1.0 / float(fs_hz)
            tau = 1.0 / (2.0 * np.pi * float(cutoff_hz))
            self.alpha = dt / (tau + dt)
        self.state = [0.0] * self.order
        self.initialized = False

    def update(self, x: float) -> float:
        x = float(x)
        if not self.initialized:
            self.state = [x] * self.order
            self.initialized = True
            return x
        y = x
        for i in range(self.order):
            self.state[i] = self.state[i] + self.alpha * (y - self.state[i])
            y = self.state[i]
        return float(y)


def apply_causal_lpf_series(x: np.ndarray, fs_hz: float, cutoff_hz: float, order: int) -> np.ndarray:
    lpf = _CausalLowPass(fs_hz, cutoff_hz, order)
    return np.asarray([lpf.update(float(v)) for v in np.asarray(x, dtype=np.float64)], dtype=np.float32)


def _fill_nan_1d(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def _tcn_ctor_kwargs(cfg: dict) -> dict:
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != 'self'}
    return {k: v for k, v in cfg.items() if k in allowed}


def load_synced_encoder_angle(stem: str, wave: Dict) -> Tuple[np.ndarray, str]:
    npz_path = TELEMETRY_ROOT / f'{stem}.npz'
    if not npz_path.is_file():
        raise FileNotFoundError(npz_path)
    d = np.load(str(npz_path), allow_pickle=True)
    if 'time' not in d.files:
        raise KeyError(f"No 'time' in {npz_path.name}; keys={sorted(d.files)}")
    t_npz = np.asarray(d['time'], dtype=np.float64)
    for key in ('model_in_knee_angle_raw', 'knee_angle_r'):
        if key in d.files:
            enc_raw = np.asarray(d[key], dtype=np.float64)
            enc_sync = sync_to_wave_t(t_npz, enc_raw, wave, t_src_on_npz_clock=True)
            return enc_sync, key
    raise KeyError(f'No encoder angle in {npz_path.name}; keys={sorted(d.files)}')


load_logged_encoder_angle = load_synced_encoder_angle


def causal_backward_derivative(x: np.ndarray, fs_hz: float) -> np.ndarray:
    """Backward Euler derivative (causal, real-time safe)."""
    arr = np.asarray(x, dtype=np.float64)
    dt = 1.0 / float(fs_hz)
    vel = np.zeros_like(arr)
    if len(arr) > 1:
        vel[1:] = (arr[1:] - arr[:-1]) / dt
    return vel


def _vicon_ik_path(stem: str) -> Path:
    paths = resolve_trial_paths(stem)
    ik_path = paths['subject_dir'] / EXO_KIND / 'ik' / f"{paths['cond']}_{paths['speed']}_ik.mot"
    if not ik_path.is_file():
        raise FileNotFoundError(ik_path)
    return ik_path


def _load_vicon_knee_ik(stem: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    cols, data = read_sto(_vicon_ik_path(stem))
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_rad = np.deg2rad(data[:, cols.index('knee_angle_r')].astype(np.float64))
    fs_mocap = infer_fs_hz(t_mocap, default_fs=100.0)
    return t_mocap, knee_rad, fs_mocap


@torch.no_grad()
def run_knee_tcn_inference(
    model: TCN,
    angle: np.ndarray,
    vel: np.ndarray,
    window_size: int,
    device: str,
) -> np.ndarray:
    angle = np.asarray(angle, dtype=np.float32)
    vel = np.asarray(vel, dtype=np.float32)
    n = int(min(len(angle), len(vel)))
    pred = np.zeros(n, dtype=np.float32)
    model.eval()
    for t in range(n):
        start = max(0, t - window_size + 1)
        valid = t - start + 1
        x = np.zeros((2, window_size), dtype=np.float32)
        x[0, -valid:] = angle[start : t + 1]
        x[1, -valid:] = vel[start : t + 1]
        xt = torch.from_numpy(x).unsqueeze(0).to(device=device, dtype=torch.float32)
        y = model(xt)
        pred[t] = float(y[0, 0, -1].item())
    return pred


def build_model_compare_wave(stem: str, wave: Dict, model: TCN, window_size: int, device: str) -> Dict:
    """Offline replay: Vicon IK inputs and encoder + causal d(encoder)/dt inputs."""
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(t_aligned)))
    out = dict(wave)

    t_mocap, knee_rad, _fs_mocap = _load_vicon_knee_ik(stem)
    angle_sync = sync_to_wave_t(t_mocap, knee_rad, wave, t_src_on_npz_clock=False)
    knee_vel_mocap = np.gradient(knee_rad, 1.0 / _fs_mocap)
    vel_sync = sync_to_wave_t(t_mocap, knee_vel_mocap, wave, t_src_on_npz_clock=False)
    enc_sync, enc_key = load_synced_encoder_angle(stem, wave)
    n = int(min(len(angle_sync), len(vel_sync), len(enc_sync), len(out.get('gt_nmpkg', t_aligned))))
    angle_sync = angle_sync[:n]
    vel_sync = vel_sync[:n]
    enc_sync = enc_sync[:n]
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(t_aligned[:n])))

    vicon_angle = apply_causal_lpf_series(angle_sync, fs_hz, VICON_ANGLE_LPF_HZ, VICON_ANGLE_LPF_ORDER)
    vicon_vel = apply_causal_lpf_series(vel_sync, fs_hz, VICON_VEL_LPF_HZ, VICON_VEL_LPF_ORDER)
    vicon_pred_raw = run_knee_tcn_inference(model, vicon_angle, vicon_vel, window_size, device).astype(np.float64)
    out['vicon_ik_model_out_nmpkg_raw'] = vicon_pred_raw
    out['vicon_ik_model_out_nmpkg'] = model_out_nmpkg_lpf(vicon_pred_raw, fs_hz)

    enc_angle = apply_causal_lpf_series(enc_sync, fs_hz, VICON_ANGLE_LPF_HZ, VICON_ANGLE_LPF_ORDER)
    enc_vel_raw = causal_backward_derivative(enc_angle, fs_hz)
    enc_vel = apply_causal_lpf_series(enc_vel_raw, fs_hz, VICON_VEL_LPF_HZ, VICON_VEL_LPF_ORDER)
    enc_pred_raw = run_knee_tcn_inference(model, enc_angle, enc_vel, window_size, device).astype(np.float64)
    out['encoder_angle_key'] = enc_key
    out['encoder_replay_model_out_nmpkg_raw'] = enc_pred_raw
    out['encoder_replay_model_out_nmpkg'] = model_out_nmpkg_lpf(enc_pred_raw, fs_hz)

    for key in ('t', 'gt_nmpkg', 'model_out_nmpkg', 'model_out_nmpkg_raw'):
        if key in out and len(out[key]) > n:
            out[key] = np.asarray(out[key], dtype=np.float64)[:n]

    return out


print(f'Loading checkpoint: {VICON_ORACLE_CKPT}')
if not VICON_ORACLE_CKPT.is_file():
    raise FileNotFoundError(VICON_ORACLE_CKPT)

_ckpt = torch.load(str(VICON_ORACLE_CKPT), map_location='cpu', weights_only=False)
_model_cfg = _ckpt['model_config']
if _model_cfg.get('model_type', 'tcn') != 'tcn':
    raise ValueError(f"Expected TCN checkpoint, got {_model_cfg.get('model_type')!r}")

_window_size = int(_ckpt.get('window_size', _ctrl_cfg.get('frame_length', 100)))
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_replay_model = TCN(**_tcn_ctor_kwargs(_model_cfg)).eval()
_replay_model.load_state_dict(_ckpt['model_state_dict'])
_replay_model.to(_device)
print(
    f'Model replay ready | window={_window_size} | device={_device} | '
    f'in LPF angle={VICON_ANGLE_LPF_HZ:g} Hz / {VICON_ANGLE_LPF_ORDER}c | '
    f'vel={VICON_VEL_LPF_HZ:g} Hz / {VICON_VEL_LPF_ORDER}c | '
    f'encoder vel=causal d(angle)/dt | output={LPF_CUTOFF_HZ:g} Hz zero-phase'
)

VICON_ORACLE_DATA: Dict[str, Dict] = {}
_compare_errors = []
for stem, wave in sorted(TRIAL_DATA.items()):
    try:
        VICON_ORACLE_DATA[stem] = build_model_compare_wave(stem, wave, _replay_model, _window_size, _device)
    except Exception as exc:
        _compare_errors.append((stem, str(exc)))

print(f'Built model-compare waveforms for {len(VICON_ORACLE_DATA)} / {len(TRIAL_DATA)} trials')
if _compare_errors:
    print('Skipped:')
    for stem, msg in _compare_errors:
        print(f'  {stem}: {msg}')

_compare_metrics_rows = []
for stem, wave in sorted(VICON_ORACLE_DATA.items()):
    meta = trial_meta_from_stem(stem)
    m = analysis_trim_mask(wave['t'])
    gt = wave['gt_nmpkg'][m]
    rmse_logged, r2_logged = rmse_r2(gt, wave['model_out_nmpkg'][m])
    rmse_enc, r2_enc = rmse_r2(gt, wave['encoder_replay_model_out_nmpkg'][m])
    rmse_vicon, r2_vicon = rmse_r2(gt, wave['vicon_ik_model_out_nmpkg'][m])
    _compare_metrics_rows.append({
        'trial': stem,
        'trial_key': meta['trial_key'],
        'subject': meta['subject'],
        'task': meta['task'],
        'condition': meta['condition'],
        'rmse_logged_nmpkg': rmse_logged,
        'r2_logged_nmpkg': r2_logged,
        'rmse_encoder_replay_nmpkg': rmse_enc,
        'r2_encoder_replay_nmpkg': r2_enc,
        'rmse_vicon_oracle_nmpkg': rmse_vicon,
        'r2_vicon_oracle_nmpkg': r2_vicon,
        'encoder_angle_key': wave.get('encoder_angle_key', ''),
        'offset_s': wave['offset_s'],
    })

oracle_metrics_df = pd.DataFrame(_compare_metrics_rows)
oracle_metrics_df.to_csv(MODEL_COMPARE_METRICS_CSV, index=False)
VICON_ORACLE_METRICS_CSV = MODEL_COMPARE_METRICS_CSV
oracle_metrics_df['rmse_oracle_nmpkg'] = oracle_metrics_df['rmse_vicon_oracle_nmpkg']
oracle_metrics_df['r2_oracle_nmpkg'] = oracle_metrics_df['r2_vicon_oracle_nmpkg']

print(f'Saved model-compare metrics → {MODEL_COMPARE_METRICS_CSV}')
display(oracle_metrics_df)
for label, rmse_col, r2_col in [
    ('Logged (device)', 'rmse_logged_nmpkg', 'r2_logged_nmpkg'),
    ('Encoder replay', 'rmse_encoder_replay_nmpkg', 'r2_encoder_replay_nmpkg'),
    ('Vicon IK oracle', 'rmse_vicon_oracle_nmpkg', 'r2_vicon_oracle_nmpkg'),
]:
    print(
        f"{label:16s}: RMSE={oracle_metrics_df[rmse_col].mean():.4f} N·m/kg | "
        f"R²={oracle_metrics_df[r2_col].mean():.4f}"
    )

oracle_out = widgets.Output()
oracle_trial_dd = widgets.Dropdown(options=sorted(VICON_ORACLE_DATA), description='Trial:')
oracle_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px')
)


def draw_model_compare_residual(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    t0, t1 = t_window
    gt = wave['gt_nmpkg']
    logged = wave['model_out_nmpkg']
    enc = wave['encoder_replay_model_out_nmpkg']
    vicon = wave['vicon_ik_model_out_nmpkg']
    m = (
        analysis_trim_mask(wave['t'])
        & (t_rel >= t0)
        & (t_rel <= t1)
        & np.isfinite(gt)
        & np.isfinite(logged)
        & np.isfinite(enc)
        & np.isfinite(vicon)
    )
    rmse_logged, r2_logged = rmse_r2(gt[m], logged[m])
    rmse_enc, r2_enc = rmse_r2(gt[m], enc[m])
    rmse_vicon, r2_vicon = rmse_r2(gt[m], vicon[m])
    fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axs[0].plot(t_rel[m], gt[m], color='#1e88e5', lw=2.0, label='GT (ID/mass − cmd/mass)')
    axs[0].plot(t_rel[m], logged[m], color='#e53935', lw=1.2, ls='--', alpha=0.9, label='Logged model (device)')
    axs[0].plot(
        t_rel[m], enc[m], color='#fb8c00', lw=1.5, ls='-.', alpha=0.95,
        label='Encoder replay (angle + causal dθ/dt)',
    )
    axs[0].plot(t_rel[m], vicon[m], color='#43a047', lw=1.6, ls='-', label='Vicon IK oracle')
    axs[0].set_ylabel('N·m/kg')
    axs[0].set_title(
        f'Logged RMSE={rmse_logged:.4f} R²={r2_logged:.4f} | '
        f'Enc replay RMSE={rmse_enc:.4f} R²={r2_enc:.4f} | '
        f'Vicon RMSE={rmse_vicon:.4f} R²={r2_vicon:.4f} | offset={wave["offset_s"]:+.3f}s'
    )
    axs[0].legend(loc='upper right', fontsize=8)
    axs[0].grid(alpha=0.25)
    axs[1].plot(t_rel[m], (logged - gt)[m], color='#e53935', lw=1.2, ls='--', alpha=0.9, label='Logged − GT')
    axs[1].plot(t_rel[m], (enc - gt)[m], color='#fb8c00', lw=1.4, ls='-.', label='Encoder replay − GT')
    axs[1].plot(t_rel[m], (vicon - gt)[m], color='#43a047', lw=1.4, label='Vicon IK − GT')
    axs[1].axhline(0, color='black', ls=':')
    axs[1].set_ylabel('Residual (N·m/kg)')
    axs[1].set_xlabel('Time (s)')
    axs[1].legend(loc='upper right', fontsize=8)
    axs[1].grid(alpha=0.25)
    fig.tight_layout()
    with oracle_out:
        oracle_out.clear_output(wait=True)
        plt.show()


def _init_oracle_slider(trial_key: str) -> None:
    wave = VICON_ORACLE_DATA[trial_key]
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t'])
    t_use = t_rel[m] if m.any() else t_rel
    oracle_slider.min = float(t_use[0])
    oracle_slider.max = float(t_use[-1])
    oracle_slider.step = max((oracle_slider.max - oracle_slider.min) / 500, 1e-3)
    oracle_slider.value = (oracle_slider.min, oracle_slider.max)


def _redraw_oracle(*_):
    draw_model_compare_residual(VICON_ORACLE_DATA[oracle_trial_dd.value], oracle_slider.value)


def _on_oracle_trial(change):
    _init_oracle_slider(change['new'])
    _redraw_oracle()


oracle_trial_dd.observe(_on_oracle_trial, names='value')
oracle_slider.observe(_redraw_oracle, names='value')
if VICON_ORACLE_DATA:
    _init_oracle_slider(oracle_trial_dd.value)
    display(widgets.VBox([oracle_trial_dd, oracle_slider, oracle_out]))
    _redraw_oracle()
else:
    print('No model-compare trials to plot.')


### 4b. High oracle-gain trials — encoder vs Vicon IK

Flags trials where Vicon IK → model beats the logged model by a large margin, then compares **motor encoder knee angle** (model input) vs **GPIO-synced Vicon IK**.

In [ ]:
if 'oracle_metrics_df' not in globals() or oracle_metrics_df.empty:
    raise RuntimeError('Run the §4 oracle cell first.')

ORACLE_GAIN_RMSE_MIN = 0.08   # N·m/kg improvement (logged − oracle)
ORACLE_GAIN_R2_MIN = 0.25     # R² improvement (oracle − logged)

gain_df = oracle_metrics_df.copy()
gain_df['delta_rmse_nmpkg'] = gain_df['rmse_logged_nmpkg'] - gain_df['rmse_oracle_nmpkg']
gain_df['delta_r2_nmpkg'] = gain_df['r2_oracle_nmpkg'] - gain_df['r2_logged_nmpkg']
gain_df['delta_rmse_enc_nmpkg'] = gain_df['rmse_logged_nmpkg'] - gain_df['rmse_encoder_replay_nmpkg']
gain_df['delta_r2_enc_nmpkg'] = gain_df['r2_encoder_replay_nmpkg'] - gain_df['r2_logged_nmpkg']
gain_df['oracle_gain_flag'] = (
    (gain_df['delta_rmse_nmpkg'] >= ORACLE_GAIN_RMSE_MIN)
    | (gain_df['delta_r2_nmpkg'] >= ORACLE_GAIN_R2_MIN)
)
gain_df = gain_df.sort_values(
    ['oracle_gain_flag', 'delta_rmse_nmpkg', 'delta_r2_nmpkg'],
    ascending=[False, False, False],
).reset_index(drop=True)

high_gain_df = gain_df[gain_df['oracle_gain_flag']].copy()
print(
    f'Oracle gain thresholds: ΔRMSE ≥ {ORACLE_GAIN_RMSE_MIN:.2f} N·m/kg or '
    f'ΔR² ≥ {ORACLE_GAIN_R2_MIN:.2f}'
)
print(f'Flagged {len(high_gain_df)} / {len(gain_df)} trials')
display(
    gain_df[
        [
            'trial', 'subject', 'task', 'delta_rmse_nmpkg', 'delta_r2_nmpkg',
            'rmse_logged_nmpkg', 'rmse_encoder_replay_nmpkg', 'rmse_vicon_oracle_nmpkg',
            'r2_logged_nmpkg', 'r2_encoder_replay_nmpkg', 'r2_vicon_oracle_nmpkg',
            'oracle_gain_flag',
        ]
    ]
)


if "sync_to_wave_t" not in globals():
    raise RuntimeError("Run the helpers cell first (defines sync_to_wave_t).")
if "load_synced_encoder_angle" not in globals():
    raise RuntimeError("Run §4 first (defines load_synced_encoder_angle).")
if "_load_vicon_knee_ik" not in globals():
    raise RuntimeError("Run §4 first (defines _load_vicon_knee_ik).")

def load_synced_vicon_knee_angle(stem: str, wave: Dict) -> Tuple[np.ndarray, np.ndarray]:
    t_mocap, knee_rad, _fs_mocap = _load_vicon_knee_ik(stem)
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    vicon_rad = sync_to_wave_t(t_mocap, knee_rad, wave, t_src_on_npz_clock=False)
    n = int(min(len(t_aligned), len(vicon_rad)))
    return vicon_rad[:n], t_aligned[:n]


def angle_metrics_deg(encoder_rad: np.ndarray, vicon_rad: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    enc = np.rad2deg(np.asarray(encoder_rad, dtype=np.float64)[mask])
    vic = np.rad2deg(np.asarray(vicon_rad, dtype=np.float64)[mask])
    m = np.isfinite(enc) & np.isfinite(vic)
    if m.sum() < 2:
        return {'n': int(m.sum()), 'rmse_deg': np.nan, 'r2_deg': np.nan, 'bias_deg': np.nan}
    err = enc[m] - vic[m]
    rmse = float(np.sqrt(np.mean(err ** 2)))
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((vic[m] - np.mean(vic[m])) ** 2))
    r2 = float(1.0 - ss_res / (ss_tot + 1e-12))
    return {
        'n': int(m.sum()),
        'rmse_deg': rmse,
        'r2_deg': r2,
        'bias_deg': float(np.mean(err)),
    }


IK_COMPARE_DATA: Dict[str, Dict] = {}
ik_compare_rows = []
for stem, wave in sorted(VICON_ORACLE_DATA.items()):
    row = gain_df.loc[gain_df['trial'] == stem].iloc[0]
    enc_rad, enc_key = load_synced_encoder_angle(stem, wave)
    vicon_rad, t = load_synced_vicon_knee_angle(stem, wave)
    n = min(len(t), len(enc_rad), len(vicon_rad))
    t = t[:n]
    enc_rad = enc_rad[:n]
    vicon_rad = vicon_rad[:n]
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(t)))
    m = analysis_trim_mask(t)
    enc_lpf = apply_causal_lpf_series(enc_rad, fs_hz, VICON_ANGLE_LPF_HZ, VICON_ANGLE_LPF_ORDER)
    vic_lpf = apply_causal_lpf_series(vicon_rad, fs_hz, VICON_ANGLE_LPF_HZ, VICON_ANGLE_LPF_ORDER)
    met_raw = angle_metrics_deg(enc_rad, vicon_rad, m)
    met_lpf = angle_metrics_deg(enc_lpf, vic_lpf, m)
    IK_COMPARE_DATA[stem] = {
        't': t,
        'encoder_rad': enc_rad,
        'vicon_rad': vicon_rad,
        'encoder_lpf': enc_lpf,
        'vicon_lpf': vic_lpf,
        'encoder_key': enc_key,
        'oracle_gain_flag': bool(row['oracle_gain_flag']),
        'delta_rmse_nmpkg': float(row['delta_rmse_nmpkg']),
        'delta_r2_nmpkg': float(row['delta_r2_nmpkg']),
        'metrics_raw': met_raw,
        'metrics_lpf': met_lpf,
    }
    ik_compare_rows.append({
        'trial': stem,
        'subject': row['subject'],
        'task': row['task'],
        'oracle_gain_flag': bool(row['oracle_gain_flag']),
        'delta_rmse_nmpkg': float(row['delta_rmse_nmpkg']),
        'delta_r2_nmpkg': float(row['delta_r2_nmpkg']),
        'rmse_encoder_replay_nmpkg': float(row['rmse_encoder_replay_nmpkg']),
        'r2_encoder_replay_nmpkg': float(row['r2_encoder_replay_nmpkg']),
        'enc_key': enc_key,
        'angle_rmse_deg_raw': met_raw['rmse_deg'],
        'angle_r2_raw': met_raw['r2_deg'],
        'angle_bias_deg_raw': met_raw['bias_deg'],
        'angle_rmse_deg_lpf': met_lpf['rmse_deg'],
        'angle_r2_lpf': met_lpf['r2_deg'],
        'angle_bias_deg_lpf': met_lpf['bias_deg'],
    })

ik_compare_df = pd.DataFrame(ik_compare_rows).sort_values(
    ['oracle_gain_flag', 'delta_rmse_nmpkg', 'delta_r2_nmpkg'],
    ascending=[False, False, False],
).reset_index(drop=True)
print('\nEncoder vs Vicon IK (trimmed window):')
display(ik_compare_df)

ik_out = widgets.Output()
ik_trial_dd = widgets.Dropdown(
    options=[
        (
            f"{r['trial']} [{'HIGH' if r['oracle_gain_flag'] else 'ok'} | "
            f"ΔRMSE={r['delta_rmse_nmpkg']:+.3f} ΔR²={r['delta_r2_nmpkg']:+.3f} | "
            f"angle RMSE={r['angle_rmse_deg_lpf']:.1f}°]",
            r['trial'],
        )
        for _, r in ik_compare_df.iterrows()
    ],
    description='Trial:',
    layout=widgets.Layout(width='98%'),
)
ik_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px')
)
ik_signal_dd = widgets.Dropdown(
    options=[('Raw', 'raw'), ('Causal LPF (6 Hz, device)', 'lpf')],
    value='lpf',
    description='Angles:',
)


def draw_encoder_vs_vicon_ik(stem: str, t_window: Tuple[float, float], signal_mode: str) -> None:
    d = IK_COMPARE_DATA[stem]
    t_rel = d['t'] - np.nanmin(d['t'])
    t0, t1 = t_window
    m = analysis_trim_mask(d['t']) & (t_rel >= t0) & (t_rel <= t1)
    if signal_mode == 'lpf':
        enc = np.rad2deg(d['encoder_lpf'])
        vic = np.rad2deg(d['vicon_lpf'])
        sig_label = 'causal 6 Hz LPF'
        met = d['metrics_lpf']
    else:
        enc = np.rad2deg(d['encoder_rad'])
        vic = np.rad2deg(d['vicon_rad'])
        sig_label = 'raw'
        met = d['metrics_raw']
    mm = m & np.isfinite(enc) & np.isfinite(vic)
    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axs[0].plot(t_rel[mm], vic[mm], color='#1e88e5', lw=2.0, label='Vicon IK knee')
    axs[0].plot(t_rel[mm], enc[mm], color='#e53935', lw=1.4, ls='--', label=f"Encoder ({d['encoder_key']}, {sig_label})")
    axs[0].set_ylabel('Knee angle (deg)')
    gain_tag = 'HIGH oracle gain' if d['oracle_gain_flag'] else 'modest oracle gain'
    axs[0].set_title(
        f"{stem} | {gain_tag} | model ΔRMSE={d['delta_rmse_nmpkg']:+.3f} ΔR²={d['delta_r2_nmpkg']:+.3f} | "
        f"angle RMSE={met['rmse_deg']:.2f}° R²={met['r2_deg']:.3f} bias={met['bias_deg']:+.2f}° ({sig_label})"
    )
    axs[0].legend()
    axs[0].grid(alpha=0.25)
    axs[1].plot(t_rel[mm], (enc - vic)[mm], color='#9c27b0', lw=1.4, label='Encoder − Vicon IK')
    axs[1].axhline(0, color='black', ls=':')
    axs[1].set_ylabel('Residual (deg)')
    axs[1].set_xlabel('Time (s)')
    axs[1].legend()
    axs[1].grid(alpha=0.25)
    fig.tight_layout()
    with ik_out:
        ik_out.clear_output(wait=True)
        plt.show()


def _init_ik_slider(stem: str) -> None:
    t_rel = IK_COMPARE_DATA[stem]['t'] - np.nanmin(IK_COMPARE_DATA[stem]['t'])
    m = analysis_trim_mask(IK_COMPARE_DATA[stem]['t'])
    t_use = t_rel[m] if m.any() else t_rel
    ik_slider.min = float(t_use[0])
    ik_slider.max = float(t_use[-1])
    ik_slider.step = max((ik_slider.max - ik_slider.min) / 500, 1e-3)
    ik_slider.value = (ik_slider.min, ik_slider.max)


def _redraw_ik(*_):
    draw_encoder_vs_vicon_ik(ik_trial_dd.value, ik_slider.value, ik_signal_dd.value)


def _on_ik_trial(change):
    _init_ik_slider(change['new'])
    _redraw_ik()


ik_trial_dd.observe(_on_ik_trial, names='value')
ik_slider.observe(_redraw_ik, names='value')
ik_signal_dd.observe(_redraw_ik, names='value')

if high_gain_df.empty:
    print('No trials met the oracle-gain thresholds.')
else:
    print('\nHigh-gain trials (encoder vs Vicon IK):')
    for _, r in ik_compare_df[ik_compare_df['oracle_gain_flag']].iterrows():
        print(
            f"  {r['trial']}: model ΔRMSE={r['delta_rmse_nmpkg']:+.3f} ΔR²={r['delta_r2_nmpkg']:+.3f} | "
            f"angle RMSE={r['angle_rmse_deg_lpf']:.1f}° R²={r['angle_r2_lpf']:.3f}"
        )

if IK_COMPARE_DATA:
    default_stem = (
        high_gain_df.iloc[0]['trial'] if not high_gain_df.empty else ik_compare_df.iloc[0]['trial']
    )
    ik_trial_dd.value = default_stem
    _init_ik_slider(default_stem)
    display(widgets.VBox([ik_trial_dd, widgets.HBox([ik_signal_dd, ik_slider]), ik_out]))
    _redraw_ik()


## 5. Paper-ready outputs

Exports tables (CSV + LaTeX) and figures (PDF + PNG @ 300 dpi) to `analysis/paper_outputs/knee_exo/`.

In [ ]:
if not TRIAL_DATA:
    raise RuntimeError('No trials loaded.')

PAPER_EXCLUDE_SUBJECTS = {'AB02_Oscar'}
PAPER_EXCLUDE_TRIALS = {'AB05_Maria::LG_0p8mps'}
OUT_DIR = PROJECT_ROOT / 'analysis' / 'paper_outputs' / PAPER_SUBDIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAPER_RC = {
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
}
PALETTE = {'GT': '#1e88e5', 'Model': '#e53935', 'RMSE': '#3949ab', 'R2': '#43a047'}
TASK_ORDER = ['LG', 'RA', 'RD']


def _paper_exclude(trial_key: str) -> bool:
    subject = trial_key.split('::', 1)[0]
    if EXCLUDE_SUBJECTS is not None and subject in EXCLUDE_SUBJECTS:
        return True
    return subject in PAPER_EXCLUDE_SUBJECTS or trial_key in PAPER_EXCLUDE_TRIALS


def _mean_std(series, decimals=3):
    x = np.asarray(series, dtype=np.float64)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return '—'
    return f'{np.mean(x):.{decimals}f} ± {np.std(x, ddof=1):.{decimals}f}'


def _save_table(df, stem, caption=''):
    csv_path = OUT_DIR / f'{stem}.csv'
    tex_path = OUT_DIR / f'{stem}.tex'
    df.to_csv(csv_path, index=False)
    tex = df.to_latex(index=False, escape=True, caption=caption, label=f'tab:{stem}')
    tex_path.write_text(tex)
    return csv_path, tex_path


def _trial_metrics_row(stem: str, wave: Dict) -> Dict:
    meta = trial_meta_from_stem(stem)
    m = analysis_trim_mask(wave['t'])
    rmse, r2 = rmse_r2(wave['gt_nmpkg'][m], wave['model_out_nmpkg'][m])
    return {
        'trial': stem,
        'trial_key': meta['trial_key'],
        'subject': meta['subject'],
        'condition': meta['condition'],
        'task': meta['task'],
        'speed_mps': meta['speed_mps'],
        'joint': wave['joint'],
        'n_samples': int(m.sum()),
        'rmse_nmpkg': rmse,
        'r2_nmpkg': r2,
        'offset_s': wave['offset_s'],
        'mass_kg': wave['mass_kg'],
        'applied_key': wave['applied_key'],
        'model_out_key': wave['model_out_key'],
    }


metrics_rows = [_trial_metrics_row(stem, wave) for stem, wave in sorted(TRIAL_DATA.items())]
metrics_df = pd.DataFrame(metrics_rows)
metrics_df_paper = metrics_df[~metrics_df['trial_key'].map(_paper_exclude)].copy()

table_summary = pd.DataFrame([{
    'joint': JOINT_LABEL,
    'n_trials': metrics_df_paper['trial'].nunique(),
    'rmse_nmpkg_mean_std': _mean_std(metrics_df_paper['rmse_nmpkg']),
    'r2_nmpkg_mean_std': _mean_std(metrics_df_paper['r2_nmpkg']),
    'rmse_nmpkg_mean': float(metrics_df_paper['rmse_nmpkg'].mean()),
    'rmse_nmpkg_std': float(metrics_df_paper['rmse_nmpkg'].std(ddof=1)),
    'r2_nmpkg_mean': float(metrics_df_paper['r2_nmpkg'].mean()),
    'r2_nmpkg_std': float(metrics_df_paper['r2_nmpkg'].std(ddof=1)),
}])

table_per_trial = metrics_df_paper[[
    'subject', 'condition', 'task', 'speed_mps', 'n_samples',
    'rmse_nmpkg', 'r2_nmpkg', 'offset_s', 'mass_kg',
]].sort_values(['subject', 'task', 'speed_mps']).round(4)

table_overall = pd.DataFrame([{
    'metric': GT_METRIC_LABEL,
    'unit': 'N·m/kg / R²',
    'rmse_mean_std': _mean_std(metrics_df_paper['rmse_nmpkg']),
    'r2_mean_std': _mean_std(metrics_df_paper['r2_nmpkg']),
    'n_trials': len(metrics_df_paper),
}])

for stem, df, cap in [
    ('table1_summary', table_summary, f'{JOINT_LABEL} exo GT vs logged model (GPIO sync, {TRIM_START_SEC:.0f}s start + {TRIM_END_SEC:.0f}s end trim).'),
    ('table2_per_trial', table_per_trial, 'Per-trial metrics.'),
    ('table3_overall', table_overall, 'Overall summary across paper trials.'),
]:
    _save_table(df, stem, caption=cap)

print(f'Paper outputs → {OUT_DIR}')
print(f'  Sync: gpio_falling_edge | trim: {TRIM_START_SEC:.0f}s start + {TRIM_END_SEC:.0f}s end | trials: {len(metrics_df_paper)}')
display(table_summary)
display(table_per_trial.head(12))

task_order = [t for t in TASK_ORDER if t in metrics_df_paper['task'].unique()]
with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
    axes[0].bar([0], [table_summary['rmse_nmpkg_mean'].iloc[0]],
                yerr=[table_summary['rmse_nmpkg_std'].iloc[0]],
                color=PALETTE['RMSE'], capsize=4, alpha=0.9, width=0.55)
    axes[0].set_xticks([0]); axes[0].set_xticklabels([JOINT_LABEL])
    axes[0].set_ylabel('RMSE (N·m/kg)'); axes[0].set_title('(a) RMSE')
    axes[1].bar([0], [table_summary['r2_nmpkg_mean'].iloc[0]],
                yerr=[table_summary['r2_nmpkg_std'].iloc[0]],
                color=PALETTE['R2'], capsize=4, alpha=0.9, width=0.55)
    axes[1].set_xticks([0]); axes[1].set_xticklabels([JOINT_LABEL])
    axes[1].set_ylabel('R²'); axes[1].set_ylim(0, 1.05); axes[1].set_title('(b) R²')
    fig.suptitle(f'{JOINT_LABEL} exo: GT vs logged model (n={len(metrics_df_paper)} trials)', y=1.02, fontsize=12)
    fig.tight_layout()
    for ext in ('pdf', 'png'):
        fig.savefig(OUT_DIR / f'fig1_summary.{ext}')
    plt.show()

    if len(task_order) >= 2:
        groups = [metrics_df_paper.loc[metrics_df_paper['task'] == t, 'r2_nmpkg'].to_numpy() for t in task_order]
        fig, ax = plt.subplots(figsize=(4.8, 3.2))
        bp = ax.boxplot(groups, tick_labels=task_order, patch_artist=True)
        for box in bp['boxes']:
            box.set(facecolor=PALETTE['GT'], alpha=0.55)
        ax.set_ylabel('R²'); ax.set_xlabel('Task')
        ax.set_title('Trial-level R² by walking task')
        ax.set_ylim(0, 1.05)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(OUT_DIR / f'fig2_r2_by_task.{ext}')
        plt.show()

    ex_idx = (metrics_df_paper['r2_nmpkg'] - metrics_df_paper['r2_nmpkg'].median()).abs().argsort().iloc[0]
    ex_stem = metrics_df_paper.iloc[ex_idx]['trial']
    ex_row = metrics_df_paper.iloc[ex_idx]
    wave = TRIAL_DATA[ex_stem]
    m = analysis_trim_mask(wave['t'])
    t = wave['t'][m] - np.nanmin(wave['t'][m])
    gt = wave['gt_nmpkg'][m]
    model = wave['model_out_nmpkg'][m]
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    ax.plot(t, gt, color=PALETTE['GT'], lw=1.8, label='GT')
    ax.plot(t, model, color=PALETTE['Model'], lw=1.4, ls='--', label='Logged model (6 Hz LPF)')
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('N·m/kg')
    ax.set_title(
        f'Exemplar: {ex_row["subject"]} {ex_row["condition"]} | '
        f'R²={ex_row["r2_nmpkg"]:.2f} | offset={ex_row["offset_s"]:+.2f}s'
    )
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    fig.tight_layout()
    for ext in ('pdf', 'png'):
        fig.savefig(OUT_DIR / f'fig3_exemplar_timeseries.{ext}')
    plt.show()

print('\nSaved files:')
for p in sorted(OUT_DIR.glob('*')):
    print(f'  {p.name}')
